In [4]:
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer

# 1. Define Tasks across the 4 Cognitive Levels, including known (Wikipedia) and Evolving (Recent News/Novels) sources.
evaluation_tasks = [
    {"id": "1-1", "level": "KM", "type": "Known", "task": "Memorization: Poet Occupation"},
    {"id": "1-3", "level": "KM", "type": "Evolving", "task": "ETM: Recent Event Fact"},
    {"id": "2-4", "level": "KU", "type": "Known", "task": "NER: Entity Classification"},
    {"id": "3-1", "level": "KA", "type": "Known", "task": "Multi-hop: Shared Profession"},
    {"id": "4-1", "level": "KC", "type": "Known", "task": "Creation: Narrative Completion"}
]

# 2. SIMULATED MODEL OUTPUTS
# Model A represents a high-performing commercial model (e.g., GPT-4)
# Model B represents a smaller open-source model [9]
reference_R = "On May 1, 2015, Marilyn Mosby announced charges against six officers."
grounding_K = "Agent: Marilyn Mosby; Patient: six police officers; Event: charges filed."

model_outputs = {
    "LLM_A": {
        "1-1": 1.0, # EM Accuracy [10]
        "1-3": 0.8,
        "2-4": 0.85, # F1 Score [11]
        "3-1": 0.9,
        "4-1": { # KC strings for Level 4 Self-Contrast [12-14]
            "T": "Marilyn Mosby filed charges against the officers in May 2015.",
            "Tk": "Marilyn Mosby announced her office filed charges against six police officers."
        }
    },
    "LLM_B": {
        "1-1": 0.4,
        "1-3": 0.2,
        "2-4": 0.5,
        "3-1": 0.3,
        "4-1": {
            "T": "The officers were sued by a lawyer in Baltimore.",
            "Tk": "Marilyn Mosby reported charges for the six police officers."
        }
    }
}

# 3. METRIC 1: SELF-CONTRAST SCORING (Level 4: KC) [5]
# Distinguishes correctly created knowledge from hallucinations [7]
def calculate_kc_score(output_dict, reference):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    T, Tk = output_dict['T'], output_dict['Tk']
    
    # x = avg( ∂(T,R), ∂(T, Tk), ∂(Tk, R) )
    d_TR = scorer.score(reference, T)['rougeL'].fmeasure
    d_TTk = scorer.score(Tk, T)['rougeL'].fmeasure # Self-contrast metric
    d_TkR = scorer.score(reference, Tk)['rougeL'].fmeasure
    
    return (d_TR + d_TTk + d_TkR) / 3

# 4. METRIC 2: STANDARDIZED OVERALL SCORING [6, 7]
#zij = (xij - mean) / std_dev | sij = 100 * (zij - min(z)) / (max(z) - min(z))
def standardize_results(raw_matrix):
    # Standardize via Z-score per task (columns)
    means = raw_matrix.mean(axis=0)
    stds = raw_matrix.std(axis=0).replace(0, 1)
    z_scores = (raw_matrix - means) / stds
    
    # Scale to 0-100 range globally
    min_z = z_scores.min().min()
    max_z = z_scores.max().max()
    return 100 * (z_scores - min_z) / (max_z - min_z)

# 5. EXECUTION: RUN EVALUATION
raw_data = []
for model_name, results in model_outputs.items():
    row = [
        results["1-1"], results["1-3"], results["2-4"], results["3-1"],
        calculate_kc_score(results["4-1"], reference_R)
    ]
    raw_data.append(row)

df_raw = pd.DataFrame(raw_data, index=model_outputs.keys(), 
                      columns=["1-1 (KM)", "1-3 (KM)", "2-4 (KU)", "3-1 (KA)", "4-1 (KC)"])

# Apply KoLA standardization to make metrics comparable [15, 16]
df_standardized = standardize_results(df_raw)

print("--- RAW SCORES (Mixed Metrics: EM, F1, Self-Contrast) ---")
print(df_raw.round(3))
print("\n--- FINAL STANDARDIZED KOLA LEADERBOARD (0-100) ---")
print(df_standardized.round(1))

--- RAW SCORES (Mixed Metrics: EM, F1, Self-Contrast) ---
         1-1 (KM)  1-3 (KM)  2-4 (KU)  3-1 (KA)  4-1 (KC)
Model_A       1.0       0.8      0.85       0.9     0.561
Model_B       0.4       0.2      0.50       0.3     0.274

--- FINAL STANDARDIZED KOLA LEADERBOARD (0-100) ---
         1-1 (KM)  1-3 (KM)  2-4 (KU)  3-1 (KA)  4-1 (KC)
Model_A     100.0     100.0     100.0     100.0     100.0
Model_B       0.0       0.0       0.0       0.0       0.0
